In [2]:

import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPRegressor
from sklearn.metrics import mean_squared_error
import warnings

warnings.filterwarnings("ignore")

n = int(input("Enter number of samples (minimum 20): "))

X = []
y = []

for i in range(n):
    print("\nSample", i + 1)
    x1 = float(input("Enter feature 1: "))
    x2 = float(input("Enter feature 2: "))
    target = float(input("Enter target value: "))

    X.append([x1, x2])
    y.append(target)

X = np.array(X)
y = np.array(y)

wolves_count = int(input("\nEnter number of wolves: "))
iterations = int(input("Enter number of iterations: "))

if n < 20 or wolves_count < 3 or iterations < 1:
    raise ValueError("Use at least 20 samples, 3 wolves and 1 iteration.")

X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.3, random_state=42
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, random_state=42
)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_val = scaler.transform(X_val)
X_test = scaler.transform(X_test)

y_scaler = StandardScaler()
y_train_scaled = y_scaler.fit_transform(
    y_train.reshape(-1, 1)
).ravel()

def decode(w):
    lr = 10 ** w[0]
    neurons = int(round(w[1]))
    batch = int(round(w[2]))
    return lr, neurons, batch

def fitness(w):
    lr, neurons, batch = decode(w)

    model = MLPRegressor(
        hidden_layer_sizes=(neurons,),
        learning_rate_init=lr,
        batch_size=batch,
        max_iter=300,
        random_state=42
    )

    model.fit(X_train, y_train_scaled)

    pred = model.predict(X_val)
    pred = y_scaler.inverse_transform(
        pred.reshape(-1, 1)
    ).ravel()

    return mean_squared_error(y_val, pred)

rng = np.random.default_rng(42)

lower = np.array([-4, 10, 2])
upper = np.array([-2, 100, min(64, len(X_train))])

wolves = rng.uniform(lower, upper, (wolves_count, 3))

best_position = None
best_score = float("inf")

for iteration in range(iterations):

    scores = np.array([fitness(w) for w in wolves])
    order = np.argsort(scores)

    if scores[order[0]] < best_score:
        best_score = scores[order[0]]
        best_position = wolves[order[0]].copy()

    alpha, beta, delta = wolves[order[:3]].copy()

    a = 2 - 2 * iteration / iterations
    new_wolves = []

    for wolf in wolves:
        positions = []

        for leader in [alpha, beta, delta]:
            r1 = rng.random(3)
            r2 = rng.random(3)

            A = 2 * a * r1 - a
            C = 2 * r2

            D = np.abs(C * leader - wolf)
            positions.append(leader - A * D)

        new_wolves.append(np.mean(positions, axis=0))

    wolves = np.clip(np.array(new_wolves), lower, upper)

scores = np.array([fitness(w) for w in wolves])
index = np.argmin(scores)

if scores[index] < best_score:
    best_position = wolves[index].copy()

lr, neurons, batch = decode(best_position)

model = MLPRegressor(
    hidden_layer_sizes=(neurons,),
    learning_rate_init=lr,
    batch_size=batch,
    max_iter=300,
    random_state=42
)

model.fit(X_train, y_train_scaled)

38
1617pred = model.predict(X_test)
pred = y_scaler.inverse_transform(
    pred.reshape(-1, 1)
).ravel()

error = mean_squared_error(y_test, pred)

print("\nOPTIMIZED HYPERPARAMETERS")
print("-------------------------")
print("Learning Rate:", round(lr, 5))
print("Hidden Neurons:", neurons)
print("Batch Size:", batch)
print("Test Mean Squared Error:", round(error, 2))
print("Optimization Completed!")


Enter number of samples (minimum 20): 20

Sample 1
Enter feature 1: 1
Enter feature 2: 2
Enter target value: 5

Sample 2
Enter feature 1: 2
Enter feature 2: 3
Enter target value: 8

Sample 3
Enter feature 1: 3
Enter feature 2: 4
Enter target value: 11

Sample 4
Enter feature 1: 4
Enter feature 2: 5
Enter target value: 14

Sample 5
Enter feature 1: 5
Enter feature 2: 


ValueError: could not convert string to float: ''